[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skarma91/logicmojo-ai-july-2026/blob/main/modules/module-3-nlp-to-transformers/03-the-transformer/code/the_transformer.ipynb)

# Class 3.3: The Transformer

The slides show the block as a diagram. Here you build one transformer block in PyTorch and watch the tensor shape at every stage, so the architecture becomes concrete numbers.

What we will cover:
- a sequence of token vectors, with sinusoidal positions added
- multi-head attention, then add and norm
- a feed-forward layer, then add and norm
- a causal mask (decoder-only) and an output head to the vocabulary
- the shape at every stage (it never changes)


## A sequence of token vectors, with sinusoidal positions

Start with token embeddings and add a sinusoidal positional encoding (the formula from the slides), so each position carries a distinct signal.

In [1]:
import torch
import torch.nn as nn

torch.manual_seed(0)
seq_len, d_model = 5, 16
token_emb = torch.randn(1, seq_len, d_model)       # pretend embeddings (batch, tokens, features)

def positional_encoding(seq_len, d_model):
    pos = torch.arange(seq_len).unsqueeze(1)
    i = torch.arange(d_model).unsqueeze(0)
    angle = pos / (10000 ** (2 * (i // 2) / d_model))   # sinusoidal, from the slides
    pe = torch.zeros(seq_len, d_model)
    pe[:, 0::2] = torch.sin(angle[:, 0::2])             # even dims: sin
    pe[:, 1::2] = torch.cos(angle[:, 1::2])             # odd dims: cos
    return pe

pe = positional_encoding(seq_len, d_model)
x = token_emb + pe                                  # add position to each token
print("positional encoding shape:", tuple(pe.shape))
print("after embeddings + positions:", tuple(x.shape))

positional encoding shape: (5, 16)
after embeddings + positions: (1, 5, 16)


## Multi-head attention, then add and norm

Attention mixes information across tokens. The residual adds the input back; LayerNorm rescales. The shape is unchanged.

In [2]:
attn = nn.MultiheadAttention(d_model, num_heads=4, batch_first=True)
norm1 = nn.LayerNorm(d_model)

attn_out, attn_weights = attn(x, x, x)        # self-attention: Q=K=V=x
print("after attention        :", tuple(attn_out.shape))
x = norm1(x + attn_out)                       # residual + norm
print("after add & norm        :", tuple(x.shape))
print("attention weights shape :", tuple(attn_weights.shape), "(each row sums to 1)")

after attention        : (1, 5, 16)
after add & norm        : (1, 5, 16)
attention weights shape : (1, 5, 5) (each row sums to 1)


## Feed-forward, then add and norm

A small per-token network: expand to 4x the width, a nonlinearity, then back. Again a residual and a norm, and again the shape holds.

In [3]:
ff = nn.Sequential(nn.Linear(d_model, 4*d_model), nn.ReLU(), nn.Linear(4*d_model, d_model))
norm2 = nn.LayerNorm(d_model)

ff_out = ff(x)
print("after feed-forward     :", tuple(ff_out.shape))
x = norm2(x + ff_out)
print("after add & norm (out) :", tuple(x.shape))

n_params = sum(p.numel() for p in list(attn.parameters()) + list(ff.parameters()))
print("parameters in this block:", n_params)

after feed-forward     : (1, 5, 16)
after add & norm (out) : (1, 5, 16)
parameters in this block: 3216


## The shape never changes

Input and output are the same shape, which is exactly why blocks stack: the output of one block is a valid input to the next. Repeat N times and you have a transformer.

In [4]:
print("in and out shapes match:", tuple(x.shape))
print("a real model is this block, repeated, plus an embedding and an output head")

in and out shapes match: (1, 5, 16)
a real model is this block, repeated, plus an embedding and an output head


## Decoder-only: a causal mask

An LLM is a decoder-only stack: a token may attend only to itself and earlier tokens. Pass a causal mask (negative infinity above the diagonal) and the attention weights become lower-triangular.

In [5]:
causal = torch.triu(torch.full((seq_len, seq_len), float("-inf")), diagonal=1)
_, masked_w = attn(x, x, x, attn_mask=causal)
print("masked weights, row 0:", masked_w[0, 0].round(decimals=2).tolist())
print("masked weights, row 2:", masked_w[0, 2].round(decimals=2).tolist())
print("each token sees only itself and earlier tokens")

masked weights, row 0: [1.0, 0.0, 0.0, 0.0, 0.0]
masked weights, row 2: [0.33000001311302185, 0.3499999940395355, 0.3100000023841858, 0.0, 0.0]
each token sees only itself and earlier tokens


## The output head

One linear layer maps each token's vector to a score for every word in the vocabulary; softmax over the vocabulary gives the next-token distribution.

In [6]:
vocab_size = 100
output_head = nn.Linear(d_model, vocab_size)

with torch.no_grad():
    logits = output_head(x)                     # (batch, tokens, vocab)
    next_token = torch.softmax(logits[0, -1], dim=-1)  # distribution for the last position
print("logits shape:", tuple(logits.shape))
print("next-token distribution size:", next_token.shape[0], "sums to", round(float(next_token.sum()), 3))
print("most likely next token id:", int(next_token.argmax()))

logits shape: (1, 5, 100)
next-token distribution size: 100 sums to 1.0
most likely next token id: 54


## Your turn

**Micro-assignment.** Six problems on the transformer's parts; see `../micro-assignment/README.md`.

**Next, class 3.4 (Using pretrained transformers):** stop building them by hand and load real ones from Hugging Face.